# 02 - 数据库设计（Database Design）
> Harvard CS50’s Intro to Databases with SQL  |  课程时间：1:17:08 – 1:49:13

这章从「一张大表」的灾难开始，教你如何用**主键和外键**把数据拆成多张关联的表。

## 学习路线

| # | 内容 |
|---|------|
| 2.1 | DISTINCT 去重 & 数据冗余的危害 |
| 2.2 | ER 图 & 关系模型（1:1, 1:N, N:M） |
| 2.3 | 主键 — 唯一标识每一行 |
| 2.4 | 外键 — 关联表与参照完整性 |

---
## 2.1 去重与数据冗余

> ⚠️ **先运行这个 cell！** 它创建一张有冗余的「大表」—— 你会发现同样的问题在电子表格里有多常见。

In [ ]:
import sqlite3

# ============================================
# 辅助函数：执行 SQL 并美化输出
# ============================================
def sql(query):
    statements = [s.strip() for s in query.strip().split(';') if s.strip()]
    if not statements:
        return
    result = None
    for stmt in statements:
        cursor = conn.execute(stmt)
        if cursor.description:
            result = cursor.fetchall()
            columns = [d[0] for d in cursor.description]
    conn.commit()
    if result is None:
        print('✅ Done')
        return
    if not result:
        print('(empty)')
        return
    col_widths = [len(c) for c in columns]
    for row in result:
        for i, val in enumerate(row):
            col_widths[i] = max(col_widths[i], len(str(val)))
    header = ' | '.join(c.ljust(col_widths[i]) for i, c in enumerate(columns))
    sep = '-+-'.join('-' * col_widths[i] for i in range(len(columns)))
    print(header)
    print(sep)
    for row in result:
        print(' | '.join(str(v).ljust(col_widths[i]) for i, v in enumerate(row)))
    print(f'\n({len(result)} row{"s" if len(result) != 1 else ""})')


conn = sqlite3.connect('books_ch2.db')
conn.execute("PRAGMA foreign_keys = ON")

# ============================================
# 建一张「坏表」：所有信息堆在一起
# ============================================
conn.executescript('''
DROP TABLE IF EXISTS flat_books;

CREATE TABLE flat_books (
    title TEXT,
    author TEXT,
    author_birth INTEGER,
    year INTEGER,
    genre TEXT,
    publisher TEXT
);

INSERT INTO flat_books VALUES
    ('The Shining',      'Stephen King', 1947, 1977, 'Horror',             'Doubleday'),
    ('It',               'Stephen King', 1947, 1986, 'Horror',             'Viking'),
    ('The Stand',        'Stephen King', 1947, 1978, 'Horror',             'Doubleday'),
    ('The Martian',      'Andy Weir',    1972, 2011, 'Science Fiction',   'Crown'),
    ('Project Hail Mary','Andy Weir',    1972, 2021, 'Science Fiction',   'Ballantine'),
    ('Dune',             'Frank Herbert', 1920, 1965, 'Science Fiction',   'Chilton'),
    ('Gone Girl',        'Gillian Flynn', 1971, 2012, 'Mystery',           'Crown'),
    ('Becoming',         'Michelle Obama',1964, 2018, 'Nonfiction',        'Crown'),
    ('Educated',         'Tara Westover', 1986, 2018, 'Nonfiction',        'Random House'),
    ('Atomic Habits',    'James Clear',   1986, 2018, 'Nonfiction',        'Avery');
''')

print('📋 flat_books 表（所有数据堆在一张表里）：')
sql('SELECT * FROM flat_books;')

### DISTINCT — 去重

当你想看「出版社有哪些」时，直接查 `publisher` 列会出现大量重复：

In [ ]:
# 不去重：一大堆重复
sql('SELECT publisher FROM flat_books;')

In [ ]:
# 去重：每个出版社只出现一次
sql('SELECT DISTINCT publisher FROM flat_books;')

In [ ]:
# DISTINCT 多列：组合去重
sql('SELECT DISTINCT author, genre FROM flat_books;')

### 数据冗余：用眼就能看出来

看上面的 `flat_books` 表：
- `Stephen King` 出现 3 次，`1947` 也出现 3 次 ← **重复！**
- `Andy Weir` 出现 2 次，`1972` 出现 2 次 ← **又是重复！**

### 冗余的三大问题（实际操作演示）

In [ ]:
# 问题 ①：更新异常（Update Anomaly）
# 把 Stephen King 的出生年份从 1947 改为 1948...
# 当你只更新了 2 行而漏了 1 行，查出来的数据就「打架」了

print('改之前：')
sql("SELECT author, author_birth FROM flat_books WHERE author = 'Stephen King';")

# 模拟：忘了更新 'The Stand' 这一行
conn.execute("UPDATE flat_books SET author_birth = 1948 WHERE title IN ('The Shining', 'It');")
conn.commit()

print('\n改之后（注意 The Stand 还是 1947！）：')
sql("SELECT title, author, author_birth FROM flat_books WHERE author = 'Stephen King';")

In [ ]:
# 问题 ②：插入异常（Insert Anomaly）
# 想加一个新作者 George Orwell，但他还没书...
# title 是 NOT NULL → 插不进去！

try:
    conn.execute("INSERT INTO flat_books (author, author_birth) VALUES ('George Orwell', 1903);")
except Exception as e:
    print(f'❌ 插入失败：{e}')
    print('因为 title 不能为空，但没有书的作者就插不进去')

In [ ]:
# 问题 ③：删除异常（Delete Anomaly）
# 假设 Andy Weir 只有一本书，删除它 → 作者信息也丢了

print('删除前 Andy Weir 的信息：')
cursor = conn.execute("SELECT * FROM flat_books WHERE author = 'Andy Weir';")
for row in cursor:
    print(f'  {row[0]:20s} | {row[1]} | born {row[2]}')

conn.execute("DELETE FROM flat_books WHERE title = 'The Martian';")
conn.commit()

print('\n删除后查 Andy Weir：')
sql("SELECT * FROM flat_books WHERE author = 'Andy Weir';")

### ✏️ 你自己试

看看 `flat_books` 表里有哪些不同的 `genre`（类型）。

In [ ]:
sql("")

---
## 2.2 关系模型 & ER 图

### 拆表：从一张大表到规范化设计

运行下面的 cell，创建一个**正确设计的数据库**：

In [ ]:
# ============================================
# 正确设计：拆成 authors + books + publishers
# ============================================
conn.executescript('''
DROP TABLE IF EXISTS flat_books;
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS authors;
DROP TABLE IF EXISTS publishers;

CREATE TABLE authors (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    birth_year INTEGER
);

CREATE TABLE publishers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);

CREATE TABLE books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    year INTEGER,
    genre TEXT,
    author_id INTEGER,
    publisher_id INTEGER,
    FOREIGN KEY (author_id) REFERENCES authors(id),
    FOREIGN KEY (publisher_id) REFERENCES publishers(id)
);

INSERT INTO authors VALUES
    (1, 'Stephen King', 1947),
    (2, 'Andy Weir', 1972),
    (3, 'Frank Herbert', 1920),
    (4, 'Gillian Flynn', 1971),
    (5, 'Michelle Obama', 1964),
    (6, 'Tara Westover', 1986),
    (7, 'James Clear', 1986);

INSERT INTO publishers VALUES
    (1, 'Doubleday'),
    (2, 'Viking'),
    (3, 'Crown'),
    (4, 'Ballantine'),
    (5, 'Chilton'),
    (6, 'Random House'),
    (7, 'Avery');

INSERT INTO books VALUES
    (1, 'The Shining',       1977, 'Horror',           1, 1),
    (2, 'It',                1986, 'Horror',           1, 2),
    (3, 'The Stand',         1978, 'Horror',           1, 1),
    (4, 'The Martian',       2011, 'Science Fiction',  2, 3),
    (5, 'Project Hail Mary', 2021, 'Science Fiction',  2, 4),
    (6, 'Dune',              1965, 'Science Fiction',  3, 5),
    (7, 'Gone Girl',         2012, 'Mystery',          4, 3),
    (8, 'Becoming',          2018, 'Nonfiction',       5, 3),
    (9, 'Educated',          2018, 'Nonfiction',       6, 6),
    (10,'Atomic Habits',     2018, 'Nonfiction',       7, 7);
''')

print('✅ 规范化数据库创建完毕')
print('\n📋 authors 表：')
sql('SELECT * FROM authors;')
print('\n📋 books 表：')
sql('SELECT * FROM books;')
print('\n📋 publishers 表：')
sql('SELECT * FROM publishers;')

### 关系类型可视化

```
authors（作者）              books（书）              publishers（出版社）
┌────┬──────────┬─────┐   ┌────┬───────────┬───┐   ┌────┬──────────────┐
│ id │ name     │birth│   │ id │ title     │a_id│   │ id │ name         │
├────┼──────────┼─────┤   ├────┼───────────┼───┤   ├────┼──────────────┤
│ 1  │ S. King  │1947 │──→│ 1  │The Shining│ 1 │──→│ 1  │ Doubleday    │
│    │          │     │   │ 2  │It         │ 1 │──→│ 2  │ Viking       │
│    │          │     │   │ 3  │The Stand  │ 1 │──→│ 1  │ Doubleday    │
│ 2  │ A. Weir  │1972 │──→│ 4  │The Martian│ 2 │──→│ 3  │ Crown        │
│    │          │     │   │ 5  │Proj H.M.  │ 2 │──→│ 4  │ Ballantine   │
└────┴──────────┴─────┘   └────┴───────────┴───┘   └────┴──────────────┘
    1 : N                           N : 1
 (一个作者多本书)              (多本书 → 一个出版社)
```

In [ ]:
# 现在没有冗余了：作者信息只存一处
# 修改 Stephen King 的出生年份？只改一行！
sql("UPDATE authors SET birth_year = 1947 WHERE name = 'Stephen King';")
print('修改后所有 Stephen King 相关的数据自动一致（因为 birth_year 只存在 authors 表）')
sql("SELECT id, name, birth_year FROM authors WHERE name = 'Stephen King';")

In [ ]:
# 新作者还没书？直接加，完美！
sql("INSERT INTO authors (name, birth_year) VALUES ('George Orwell', 1903);")
print('George Orwell 已加入 authors 表，不需要他立刻有书')
sql("SELECT * FROM authors WHERE name = 'George Orwell';")

### ✏️ 识别关系

上面这张 ER 图里：
- `authors → books` 是什么关系？（提示：一个作者写多少本书？）
- `books → publishers` 是什么关系？（提示：一本书有几个出版社？）

In [ ]:
# 👇 写你的答案（用注释即可）
# authors → books：
# books → publishers：


---
## 2.3 主键

### 你已经在用主键了

看 `authors` 表的 `id` 列：
- 每个作者有**唯一**的 id
- 不管有多少同名的人，id 永远不会冲突
- `INTEGER PRIMARY KEY` 自动递增

In [ ]:
# 查看主键：id 是唯一的
sql('SELECT id, name FROM authors;')

In [ ]:
# 主键的规则 ①：不能为 NULL
try:
    sql("INSERT INTO authors (id, name) VALUES (NULL, 'No Name');")
except Exception as e:
    print(f'❌ 主键不能为 NULL：{e}')

In [ ]:
# 主键的规则 ②：不能重复
try:
    sql("INSERT INTO authors (id, name) VALUES (1, 'Duplicate ID');")
except Exception as e:
    print(f'❌ id=1 已存在（被 Stephen King 占了）：{e}')

### 自增主键

在 SQLite 中，`INTEGER PRIMARY KEY` 如果不指定值，**自动取当前最大值 + 1**。

In [ ]:
# 不指定 id，自动分配
conn.execute("INSERT INTO authors (name, birth_year) VALUES ('J.K. Rowling', 1965);")
conn.commit()
sql("SELECT id, name, birth_year FROM authors WHERE name LIKE '%Rowling%';")
print('id 自动分配为下一个可用值')

### 自然键 vs 代理键（为什么不用 ISBN 做主键）

```sql
-- ❌ 自然键做主键
CREATE TABLE books (
    isbn TEXT PRIMARY KEY,   -- 问题：ISBN 可能变、可能重复、可能有书没 ISBN
    title TEXT
);

-- ✅ 代理键做主键，自然键只设 UNIQUE
CREATE TABLE books (
    id INTEGER PRIMARY KEY,  -- 纯粹标识行，稳定不变
    isbn TEXT UNIQUE,        -- ISBN 有 UNIQUE 约束但不当主键
    title TEXT
);
```

---
## 2.4 外键

### 外键是表之间的桥梁

`books.author_id` 引用 `authors.id`。数据库会**自动检查**：你插入的 `author_id` 必须在 `authors` 表里真实存在。

In [ ]:
# 外键约束：不能引用不存在的作者
try:
    sql("INSERT INTO books (title, author_id) VALUES ('Fake Book', 999);")
except Exception as e:
    print(f'❌ author_id=999 不存在于 authors 表中：{e}')
    print('外键约束阻止了脏数据！')

In [ ]:
# 外键也可以为 NULL（除非设置了 NOT NULL）
# 表示「这本书暂时没有作者」
sql("INSERT INTO books (title, author_id) VALUES ('Anonymous Manuscript', NULL);")
sql("SELECT * FROM books WHERE author_id IS NULL;")

### ON DELETE：删除关联数据的行为

我重建 books 表，加上 `ON DELETE CASCADE`：

In [ ]:
# 先删除旧数据，重建带 CASCADE 的表
conn.executescript('''
DELETE FROM books;
DROP TABLE IF EXISTS books;

-- 注意这个 ON DELETE CASCADE
CREATE TABLE books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    year INTEGER,
    genre TEXT,
    author_id INTEGER,
    publisher_id INTEGER,
    FOREIGN KEY (author_id) REFERENCES authors(id) ON DELETE CASCADE,
    FOREIGN KEY (publisher_id) REFERENCES publishers(id) ON DELETE SET NULL
);

INSERT INTO books VALUES
    (1, 'The Shining',       1977, 'Horror',           1, 1),
    (2, 'It',                1986, 'Horror',           1, 2),
    (3, 'The Martian',       2011, 'Science Fiction',  2, 3),
    (4, 'Project Hail Mary', 2021, 'Science Fiction',  2, 4),
    (5, 'Dune',              1965, 'Science Fiction',  3, 5);
''')

print('当前 books 表：')
sql('SELECT * FROM books;')

In [ ]:
# 演示 CASCADE：删除 Stephen King → 他的书自动删除
print('删除前 Stephen King 的书：')
sql("SELECT b.title FROM books b JOIN authors a ON b.author_id = a.id WHERE a.name = 'Stephen King';")

sql("DELETE FROM authors WHERE name = 'Stephen King';")

print('\n删除后，Stephen King 的书自动消失了（CASCADE）：')
sql("SELECT b.title FROM books b WHERE b.author_id = 1;")

print('\n当前 books 表：')
sql('SELECT * FROM books;')

In [ ]:
# 演示 SET NULL：删除出版社 → 书的 publisher_id 变成 NULL（而不是整行消失）
print('删除前 Dune 的出版社信息：')
sql('SELECT b.title, p.name AS publisher FROM books b JOIN publishers p ON b.publisher_id = p.id WHERE b.title = "Dune";')

sql("DELETE FROM publishers WHERE name = 'Chilton';")

print('\n删除后 Dune 的 publisher_id 变成 NULL：')
sql('SELECT * FROM books WHERE title = "Dune";')

### ON DELETE 行为对照

| 行为 | 效果 | 适用场景 |
|------|------|----------|
| `CASCADE` | 父删子也删 | 订单被取消 → 订单项也没意义了 |
| `SET NULL` | 父删子变 NULL | 出版社倒闭了 → 书还在，只是没出版社 |
| `RESTRICT` | 有子就不能删父（默认） | 有关联数据时阻止删除 |
| `SET DEFAULT` | 父删子变默认值 | 设一个「未知」占位值 |

### ✏️ 自己动手

给 `books` 表加一本新书，作者 `author_id = 2`（Andy Weir），书名自选。

In [ ]:
sql("")

---
## 🎯 综合挑战

用下面重建的完整数据集来回答这些问题。

In [ ]:
# 重建完整的干净数据
conn.executescript('''
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS authors;
DROP TABLE IF EXISTS publishers;

CREATE TABLE authors (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    birth_year INTEGER
);

CREATE TABLE publishers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);

CREATE TABLE books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    year INTEGER,
    genre TEXT,
    author_id INTEGER,
    publisher_id INTEGER,
    FOREIGN KEY (author_id) REFERENCES authors(id),
    FOREIGN KEY (publisher_id) REFERENCES publishers(id)
);

INSERT INTO authors VALUES
    (1, 'Stephen King', 1947),
    (2, 'Andy Weir', 1972),
    (3, 'Frank Herbert', 1920),
    (4, 'Gillian Flynn', 1971),
    (5, 'Tara Westover', 1986),
    (6, 'James Clear', 1986);

INSERT INTO publishers VALUES
    (1, 'Doubleday'), (2, 'Viking'), (3, 'Crown'),
    (4, 'Ballantine'), (5, 'Chilton'), (6, 'Random House'),
    (7, 'Avery');

INSERT INTO books VALUES
    (1, 'The Shining',       1977, 'Horror',           1, 1),
    (2, 'It',                1986, 'Horror',           1, 2),
    (3, 'The Stand',         1978, 'Horror',           1, 1),
    (4, 'The Martian',       2011, 'Science Fiction',  2, 3),
    (5, 'Project Hail Mary', 2021, 'Science Fiction',  2, 4),
    (6, 'Dune',              1965, 'Science Fiction',  3, 5),
    (7, 'Gone Girl',         2012, 'Mystery',          4, 3),
    (8, 'Educated',          2018, 'Nonfiction',       5, 6),
    (9, 'Atomic Habits',     2018, 'Nonfiction',       6, 7);
''')

print('✅ 数据就绪')
sql('SELECT * FROM authors;')
sql('SELECT * FROM books;')

In [ ]:
# Q1：books 表里有多少种不同的 genre？
sql("")

In [ ]:
# Q2：哪些作者出生于 1970 年之后？
sql("")

In [ ]:
# Q3：books 表中 author_id 的含义是什么？为什么它不能随便填？
# 👇 写你的理解


In [ ]:
# Q4：如果从 publishers 中删除 Crown（id=3），会有什么影响？
#     提示：看看哪些书属于 Crown
sql("SELECT title, publisher_id FROM books WHERE publisher_id = 3;")

---
## ✅ 检查清单

- [ ] 会用 `DISTINCT` 去重
- [ ] 理解数据冗余的三大危害：更新异常、插入异常、删除异常
- [ ] 能识别三种关系类型：一对一、一对多、多对多
- [ ] 理解 ER 图中实体和关系的基本概念
- [ ] 知道主键的作用：唯一标识每一行
- [ ] 知道为什么推荐代理键（自增 ID）而不是自然键
- [ ] 理解外键的作用：建立表之间的关联
- [ ] 知道外键保证参照完整性
- [ ] 了解 ON DELETE 的四种行为（CASCADE / SET NULL / RESTRICT / SET DEFAULT）
- [ ] 能自己设计一个简单的多表 Schema

---

> 📖 下一章：[03 - 高级查询](../03-advanced-queries/) — 子查询、JOIN、集合操作